# Read and inspect the local MOABB Liu2024 dataset

This notebook reads the **already-downloaded** EDF files directly with MNE. It does not call a MOABB downloader or access the network. The recording is opened lazily (`preload=False`) so the full signal is not loaded into memory during metadata inspection.

In [ ]:
from pathlib import Path

import mne
import numpy as np
import pandas as pd
from IPython.display import display

print(f"MNE version: {mne.__version__}")

In [ ]:
# Find the repository root whether launched from the repo root or this directory.
# No download is attempted if the data is absent.
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data/moabb/MNE-liu2024-data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Local Liu2024 data not found under data/moabb/MNE-liu2024-data. "
        "This notebook intentionally does not download it."
    )

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATASET_ROOT = REPO_ROOT / "data/moabb/MNE-liu2024-data/files"
EDF_ROOT = DATASET_ROOT / "edffile"
PARTICIPANTS_PATH = DATASET_ROOT / "participants.tsv"

edf_files = sorted(EDF_ROOT.glob("sub-*/eeg/*.edf"))
if not edf_files:
    raise FileNotFoundError(f"No local EDF files found beneath {EDF_ROOT}")

print(f"Dataset root: {DATASET_ROOT}")
print(f"EDF files found: {len(edf_files)}")
print(f"First EDF: {edf_files[0].relative_to(REPO_ROOT)}")
print(f"Last EDF:  {edf_files[-1].relative_to(REPO_ROOT)}")

## Participant metadata and local file inventory

In [ ]:
participants = pd.read_csv(PARTICIPANTS_PATH, sep="\t")
inventory = pd.DataFrame({
    "subject": [path.parents[1].name for path in edf_files],
    "file": [str(path.relative_to(REPO_ROOT)) for path in edf_files],
    "size_mib": [path.stat().st_size / 1024**2 for path in edf_files],
}).round({"size_mib": 2})

print(f"Participants in TSV: {len(participants)}")
print(f"Participant columns: {participants.columns.tolist()}")
display(participants.head())
display(inventory.head())

## Open one recording lazily with MNE

Change `SUBJECT` to inspect another locally available participant.

In [ ]:
SUBJECT = "sub-01"
matches = [path for path in edf_files if path.parents[1].name == SUBJECT]
if len(matches) != 1:
    raise ValueError(f"Expected one EDF for {SUBJECT}, found {len(matches)}")

edf_path = matches[0]
raw = mne.io.read_raw_edf(edf_path, preload=False, verbose="ERROR")
raw

In [ ]:
recording_properties = pd.Series({
    "subject": SUBJECT,
    "n_channels": raw.info["nchan"],
    "sampling_frequency_hz": raw.info["sfreq"],
    "n_samples": raw.n_times,
    "duration_seconds": raw.n_times / raw.info["sfreq"],
    "highpass_hz": raw.info["highpass"],
    "lowpass_hz": raw.info["lowpass"],
    "measurement_date": raw.info["meas_date"],
    "n_annotations": len(raw.annotations),
    "data_preloaded": raw.preload,
}, name="value")
display(recording_properties.to_frame())

In [ ]:
channel_properties = pd.DataFrame({
    "index": np.arange(len(raw.ch_names)),
    "name": raw.ch_names,
    "type_as_read": raw.get_channel_types(),
})
display(channel_properties)

annotation_table = raw.annotations.to_data_frame()
print(f"Annotations found: {len(annotation_table)}")
display(annotation_table)

## Inspect a small signal window

MNE returns physical values in volts. This reads only the first five seconds for a compact numerical check. The EDF also contains an unnamed final channel, so its distinct values are shown separately rather than assigning it a meaning during this initial inspection.

In [ ]:
window_seconds = 4
stop_sample = int(window_seconds * raw.info["sfreq"])
window, times = raw.get_data(start=0, stop=stop_sample, return_times=True)
window_summary = pd.DataFrame({
    "channel": raw.ch_names,
    "min_volts": window.min(axis=1),
    "max_volts": window.max(axis=1),
    "mean_volts": window.mean(axis=1),
    "std_volts": window.std(axis=1),
})
display(window_summary)

unnamed_indices = [i for i, name in enumerate(raw.ch_names) if not name.strip()]
for index in unnamed_indices:
    values = np.unique(raw.get_data(picks=[index])[0])
    print(f"Distinct values in unnamed channel at index {index}: {values}")

## Verify CPz across every subject

This reads only the CPz channel from each EDF and checks whether every sample is exactly zero.

In [ ]:
cpz_rows = []
for path in edf_files:
    subject_raw = mne.io.read_raw_edf(path, preload=False, verbose="ERROR")
    if "CPz" not in subject_raw.ch_names:
        raise ValueError(f"CPz is missing from {path}")
    cpz = subject_raw.get_data(picks=["CPz"])[0]
    cpz_rows.append({
        "subject": path.parents[1].name,
        "n_samples": cpz.size,
        "minimum_volts": cpz.min(),
        "maximum_volts": cpz.max(),
        "standard_deviation_volts": cpz.std(),
        "nonzero_samples": np.count_nonzero(cpz),
        "all_samples_zero": np.all(cpz == 0),
    })

cpz_verification = pd.DataFrame(cpz_rows)
display(cpz_verification)
print(f"Subjects checked: {len(cpz_verification)}")
print(f"CPz exactly zero for every subject: {cpz_verification['all_samples_zero'].all()}")
print(f"Total nonzero CPz samples: {cpz_verification['nonzero_samples'].sum()}")

## Select the 29 usable EEG channels

MNE reads every EDF signal as EEG because the file does not preserve the physiological channel types. Here we explicitly exclude the CPz reference placeholder, the two EOG channels, and the unnamed event channel.

In [ ]:
NON_EEG_CHANNELS = {"CPz", "HEOL", "HEOR", ""}
eeg_channel_names = [
    name for name in raw.ch_names if name.strip() and name not in NON_EEG_CHANNELS
]
eeg_raw = raw.copy().pick(eeg_channel_names)

assert eeg_raw.info["nchan"] == 29
assert set(eeg_raw.get_channel_types()) == {"eeg"}
print(f"Usable EEG channels ({eeg_raw.info['nchan']}):")
print(eeg_raw.ch_names)

## Preserve electrode positions and montage metadata

The EDF files do not carry the electrode montage. Read the accompanying local electrode TSV (cached by MOABB as `38516078`) and retain its original names and coordinates in `montage_metadata["source_electrodes"]`. The file has no units or coordinate-frame declaration; preserve those fields as unknown rather than treating these values as MNE head coordinates in meters.

For the existing spatial analysis below, attach the notebook's `standard_1020` MNE template with case-insensitive matching (`FP1`/`Fp1`, `FP2`/`Fp2`). This is an **estimated template geometry**, distinct from the dataset-supplied coordinates and its documented 10–10 acquisition layout. Save the actual attached positions, frame, fiducials, and template provenance under `montage_metadata["analysis_montage"]`.

When creating the preprocessed dataset later, include `montage_metadata` under `metadata["montage"]` and save `channel_names` in the same order as the signal array. If channels or the montage change during preprocessing, regenerate this metadata from the final EEG object. No dataset is exported in this section.


In [ ]:
import hashlib
import json

ELECTRODES_PATH = DATASET_ROOT / "38516078"
if not ELECTRODES_PATH.is_file():
    ELECTRODES_PATH = DATASET_ROOT / "38516078.tsv"
if not ELECTRODES_PATH.is_file():
    raise FileNotFoundError("Local Liu2024 electrode TSV is missing (38516078).")

electrode_table = pd.read_csv(ELECTRODES_PATH, sep="\t")
required_columns = {"name", "X", "Y", "Z"}
if not required_columns.issubset(electrode_table.columns):
    raise ValueError(f"Electrode TSV must contain {sorted(required_columns)}")
if electrode_table["name"].isna().any() or electrode_table["name"].duplicated().any():
    raise ValueError("Electrode names must be present and unique")
source_xyz = electrode_table[["X", "Y", "Z"]].to_numpy(dtype=float)
if not np.isfinite(source_xyz).all():
    raise ValueError("Electrode coordinates must be finite")
source_channel_positions = dict(zip(electrode_table["name"], source_xyz.tolist()))
missing_source_positions = [ch for ch in eeg_raw.ch_names if ch not in source_channel_positions]
if missing_source_positions:
    raise ValueError(f"Missing source electrode positions: {missing_source_positions}")

display(electrode_table)
print(f"Source coordinates cover all {len(eeg_raw.ch_names)} selected EEG channels.")


In [ ]:
# Keep the analysis template separate from the unmodified source coordinates.
ANALYSIS_MONTAGE_NAME = "standard_1020"
eeg_raw.set_montage(
    mne.channels.make_standard_montage(ANALYSIS_MONTAGE_NAME),
    match_case=False,
    on_missing="raise",
)
montage = eeg_raw.get_montage()
positions = montage.get_positions()
channel_positions = positions["ch_pos"]
channel_positions_m = np.asarray([channel_positions[ch] for ch in eeg_raw.ch_names])
if not np.isfinite(channel_positions_m).all() or np.any(
    np.linalg.norm(channel_positions_m, axis=1) == 0
):
    raise ValueError("Analysis montage has missing or invalid electrode positions")

montage_metadata = {
    "acquisition_system": "10-10",
    "acquisition_reference": "CPz",
    "ground": "FPz",
    "channel_names": list(eeg_raw.ch_names),
    "source_electrodes": {
        "path": str(ELECTRODES_PATH.relative_to(REPO_ROOT)),
        "url": "https://ndownloader.figshare.com/files/38516078",
        "sha256": hashlib.sha256(ELECTRODES_PATH.read_bytes()).hexdigest(),
        "units": None,
        "coordinate_frame": None,
        "note": "Units and coordinate frame are not declared in the source TSV.",
        "channel_positions": source_channel_positions,
    },
    "analysis_montage": {
        "kind": "standard_template",
        "name": ANALYSIS_MONTAGE_NAME,
        "mne_version": mne.__version__,
        "units": "m",
        "coordinate_frame": positions["coord_frame"],
        "channel_positions": {
            ch: channel_positions[ch].tolist() for ch in eeg_raw.ch_names
        },
        "fiducials": {
            key: None if positions[key] is None else positions[key].tolist()
            for key in ("nasion", "lpa", "rpa")
        },
    },
}
# Confirm the payload can be included in a future metadata.json without conversion.
_ = json.dumps(montage_metadata, allow_nan=False)
print(montage)
display(pd.DataFrame(channel_positions_m, index=eeg_raw.ch_names, columns=["x_m", "y_m", "z_m"]))


In [ ]:
from mne.preprocessing import compute_current_source_density

# Select the trial here: start_sample/stop_sample are only defined in later cells.
CSD_TRIAL_INDEX = 0  # Zero-based occurrence of the code-2 MI onset.
CSD_WINDOW_SECONDS = 4.0
csd_marker_index = next(i for i, name in enumerate(raw.ch_names) if not name.strip())
csd_marker_values = raw.get_data(picks=[csd_marker_index])[0]
csd_marker_codes = np.rint(csd_marker_values * 1e6).astype(int)
csd_onsets = np.flatnonzero(
    (csd_marker_codes == 2) & (np.r_[0, csd_marker_codes[:-1]] != 2)
)
if not 0 <= CSD_TRIAL_INDEX < len(csd_onsets):
    raise IndexError(f"CSD_TRIAL_INDEX must be between 0 and {len(csd_onsets) - 1}")
csd_start_sample = int(csd_onsets[CSD_TRIAL_INDEX])
csd_n_samples = round(CSD_WINDOW_SECONDS * eeg_raw.info["sfreq"])
csd_stop_sample = csd_start_sample + csd_n_samples
if csd_stop_sample > eeg_raw.n_times:
    raise ValueError("Selected CSD window extends beyond the recording")

# CSD requires preloaded data. Load a copy, preserving the original EEG.
# copy=False avoids a second copy inside the CSD function.
eeg_csd = compute_current_source_density(
    eeg_raw.copy().load_data(),
    sphere="auto",
    copy=False,
)
mi_window_csd = eeg_csd.get_data(
    start=csd_start_sample,
    stop=csd_stop_sample,
)
trial_eeg_csd = mi_window_csd[np.newaxis, :, :]

assert eeg_csd.ch_names == eeg_raw.ch_names
assert trial_eeg_csd.shape == (1, 29, csd_n_samples)
assert np.isfinite(trial_eeg_csd).all()
print(trial_eeg_csd.shape)  # (1, 29, 2000) for four seconds at 500 Hz


## Plot EEG frequency bands for one motor-imagery window

The marker pulses repeat every eight seconds: code 1, about two seconds later code 2, and about four seconds later code 3. Therefore, the natural task window is the **four seconds from code 2 to code 3**. Filtering is performed on the complete channel before slicing the window, which avoids filtering a short segment in isolation.

In [ ]:
import matplotlib.pyplot as plt

CHANNEL = "C3"       # Change to another usable EEG channel if desired.
TRIAL_INDEX = 0        # Zero-based occurrence of marker code 2.
WINDOW_SECONDS = 4.0  # Code 2 to code 3 in this recording protocol.

marker_index = next(i for i, name in enumerate(raw.ch_names) if not name.strip())
marker_values = raw.get_data(picks=[marker_index])[0]
# EDF physical scaling represents marker integers 0, 1, 2, and 3 as micro-units.
marker_codes = np.rint(marker_values * 1e6).astype(int)
code_2_onsets = np.flatnonzero((marker_codes == 2) & (np.r_[0, marker_codes[:-1]] != 2))
if TRIAL_INDEX >= len(code_2_onsets):
    raise IndexError(f"TRIAL_INDEX must be below {len(code_2_onsets)}")

start_sample = code_2_onsets[TRIAL_INDEX]
stop_sample = start_sample + round(WINDOW_SECONDS * raw.info["sfreq"])
if CHANNEL not in eeg_raw.ch_names:
    raise ValueError(f"{CHANNEL!r} is not one of the 29 usable EEG channels")
channel_data = eeg_raw.get_data(picks=[CHANNEL])[0]
bands = {
    "Delta (1–4 Hz)": (1, 4),
    "Theta (4–8 Hz)": (4, 8),
    "Alpha (8–13 Hz)": (8, 13),
    "Beta (13–30 Hz)": (13, 30),
    "Gamma (30–40 Hz)": (30, 40),
}
window_times = np.arange(stop_sample - start_sample) / raw.info["sfreq"]

fig, axes = plt.subplots(len(bands), 1, figsize=(12, 10), sharex=True)
for axis, (band_name, (low_hz, high_hz)) in zip(axes, bands.items()):
    filtered = mne.filter.filter_data(
        channel_data, raw.info["sfreq"], low_hz, high_hz, verbose="ERROR"
    )
    axis.plot(window_times, filtered[start_sample:stop_sample] * 1e6, linewidth=0.9)
    axis.set_ylabel("µV")
    axis.set_title(band_name, loc="left")
    axis.grid(alpha=0.25)

axes[-1].set_xlabel("Time from code-2 onset (seconds)")
fig.suptitle(f"{SUBJECT} — {CHANNEL} — trial {TRIAL_INDEX + 1} — {WINDOW_SECONDS:g}-second task window")
fig.tight_layout()
plt.show()

print(f"Code-2 markers found: {len(code_2_onsets)}")
print(f"Selected window: {start_sample / raw.info['sfreq']:.3f}–{stop_sample / raw.info['sfreq']:.3f} s")

In [ ]:
import matplotlib.pyplot as plt

CHANNEL = "C3"       # Change to another usable EEG channel if desired.
TRIAL_INDEX = 1        # Zero-based occurrence of marker code 2.
WINDOW_SECONDS = 4.0  # Code 2 to code 3 in this recording protocol.

marker_index = next(i for i, name in enumerate(raw.ch_names) if not name.strip())
marker_values = raw.get_data(picks=[marker_index])[0]
# EDF physical scaling represents marker integers 0, 1, 2, and 3 as micro-units.
marker_codes = np.rint(marker_values * 1e6).astype(int)
code_2_onsets = np.flatnonzero((marker_codes == 2) & (np.r_[0, marker_codes[:-1]] != 2))
if TRIAL_INDEX >= len(code_2_onsets):
    raise IndexError(f"TRIAL_INDEX must be below {len(code_2_onsets)}")

start_sample = code_2_onsets[TRIAL_INDEX]
stop_sample = start_sample + round(WINDOW_SECONDS * raw.info["sfreq"])
if CHANNEL not in eeg_raw.ch_names:
    raise ValueError(f"{CHANNEL!r} is not one of the 29 usable EEG channels")
channel_data = eeg_raw.get_data(picks=[CHANNEL])[0]
bands = {
    "Delta (1–4 Hz)": (1, 4),
    "Theta (4–8 Hz)": (4, 8),
    "Alpha (8–13 Hz)": (8, 13),
    "Beta (13–30 Hz)": (13, 30),
    "Gamma (30–40 Hz)": (30, 40),
}
window_times = np.arange(stop_sample - start_sample) / raw.info["sfreq"]

fig, axes = plt.subplots(len(bands), 1, figsize=(12, 10), sharex=True)
for axis, (band_name, (low_hz, high_hz)) in zip(axes, bands.items()):
    filtered = mne.filter.filter_data(
        channel_data, raw.info["sfreq"], low_hz, high_hz, verbose="ERROR"
    )
    axis.plot(window_times, filtered[start_sample:stop_sample] * 1e6, linewidth=0.9)
    axis.set_ylabel("µV")
    axis.set_title(band_name, loc="left")
    axis.grid(alpha=0.25)

axes[-1].set_xlabel("Time from code-2 onset (seconds)")
fig.suptitle(f"{SUBJECT} — {CHANNEL} — trial {TRIAL_INDEX + 1} — {WINDOW_SECONDS:g}-second task window")
fig.tight_layout()
plt.show()

print(f"Code-2 markers found: {len(code_2_onsets)}")
print(f"Selected window: {start_sample / raw.info['sfreq']:.3f}–{stop_sample / raw.info['sfreq']:.3f} s")

## Create a 29-channel × 5-band feature matrix

For each motor-imagery trial, Welch's method estimates the power spectral density (PSD) of each of the 29 usable EEG channels. Integrating the PSD over each frequency band gives the absolute band power $P_{i,b}$ in $\mu\mathrm{V}^2$. The powers are then converted to decibels relative to $1\,\mu\mathrm{V}^2$.

For channel $i$, the band power is

$$
P_{i,b} = \int_{f_{\mathrm{low},b}}^{f_{\mathrm{high},b}} S_i(f)\,df,
$$

where $S_i(f)$ is the Welch-estimated PSD in $\mu\mathrm{V}^2/\mathrm{Hz}$. The node-feature vector for EEG channel $i$ is defined as

$$
\mathbf{x}_i = \left[10\log_{10}\left(\frac{P_{i,b}}{1\,\mu\mathrm{V}^2}\right)\right]_{b \in \{\delta,\theta,\alpha,\beta,\gamma\}} \in \mathbb{R}^5.
$$

Stacking the 29 channel feature vectors produces the node-feature matrix for one trial:

$$
\mathbf{X} = \begin{bmatrix}\mathbf{x}_1^\top \\ \mathbf{x}_2^\top \\ \vdots \\ \mathbf{x}_{29}^\top\end{bmatrix} \in \mathbb{R}^{29 \times 5}.
$$

In [ ]:
from scipy.integrate import simpson
from scipy.signal import welch

FEATURE_TRIAL_INDEX = 1
FEATURE_BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 40),
}

code_3_onsets = np.flatnonzero(
    (marker_codes == 3) & (np.r_[0, marker_codes[:-1]] != 3)
)
feature_start = code_2_onsets[FEATURE_TRIAL_INDEX]
later_code_3 = code_3_onsets[code_3_onsets > feature_start]
if later_code_3.size == 0:
    raise ValueError("No code-3 endpoint follows the selected code-2 marker")
feature_stop = later_code_3[0]
mi_window = eeg_raw.get_data(start=feature_start, stop=feature_stop)

# Two-second Welch segments with 50% overlap give 0.5-Hz resolution at 500 Hz.
nperseg = min(round(2 * eeg_raw.info["sfreq"]), mi_window.shape[1])
frequencies, psd = welch(
    mi_window,
    fs=eeg_raw.info["sfreq"],
    nperseg=nperseg,
    noverlap=nperseg // 2,
    detrend="constant",
    axis=-1,
)

band_power = {}
for band_name, (low_hz, high_hz) in FEATURE_BANDS.items():
    mask = (frequencies >= low_hz) & (frequencies <= high_hz)
    # Welch PSD is V²/Hz; integration gives V², then convert to µV².
    band_power[band_name] = simpson(psd[:, mask], x=frequencies[mask], axis=-1) * 1e12

feature_matrix = pd.DataFrame(band_power, index=eeg_raw.ch_names)
feature_matrix.index.name = "channel"

assert feature_matrix.shape == (29, 5)

# Convert absolute power to dB relative to 1 µV². Keep the linear matrix too.
EPS = 1e-12
db_feature_matrix = 10 * np.log10(feature_matrix.clip(lower=EPS))
assert db_feature_matrix.shape == (29, 5)

print("Absolute band power (µV²):")
display(feature_matrix.round(4))
print("Band power (dB relative to 1 µV²):")
display(db_feature_matrix.round(4))
print(f"Feature matrix shape: {db_feature_matrix.shape}")
print(
    f"MI window: {feature_start / eeg_raw.info['sfreq']:.3f}–"
    f"{feature_stop / eeg_raw.info['sfreq']:.3f} s "
    f"({(feature_stop - feature_start) / eeg_raw.info['sfreq']:.3f} s)"
)

## Plot band-power profiles in stacked panels

This five-panel figure visualizes the original log-band-power matrix before spectral entropy is appended. Each point is one channel's summarized power for the selected MI window, not a time-domain EEG sample.

In [ ]:
band_channels = db_feature_matrix.index.to_list()
band_channel_positions = np.arange(len(band_channels))

fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
for axis, band_name in zip(axes, FEATURE_BANDS):
    values = db_feature_matrix[band_name].to_numpy()
    axis.plot(band_channel_positions, values, marker="o", linewidth=1.2, markersize=4)
    axis.fill_between(band_channel_positions, values, values.min(), alpha=0.15)
    axis.set_ylabel("dB")
    low_hz, high_hz = FEATURE_BANDS[band_name]
    axis.set_title(f"{band_name.title()} ({low_hz}–{high_hz} Hz)", loc="left")
    axis.grid(alpha=0.25)
axes[-1].set_xticks(band_channel_positions)
axes[-1].set_xticklabels(band_channels, rotation=60, ha="right")
axes[-1].set_xlabel("EEG channel")
fig.suptitle(f"{SUBJECT} — trial {FEATURE_TRIAL_INDEX + 1} — five band-power profiles")
fig.tight_layout()
plt.show()

## Add spectral entropy and broadband Hjorth features

Five band powers plus normalized 1–40 Hz entropy, Hjorth mobility (per sample), and complexity (dimensionless) form eight features per electrode per trial. Hjorth uses the original EEG trial without additional band filtering; CSD generation calculates the same descriptors from its own CSD trial.


In [ ]:
from scipy.stats import entropy

ENTROPY_FMIN = 1.0
ENTROPY_FMAX = 40.0
entropy_mask = (frequencies >= ENTROPY_FMIN) & (frequencies <= ENTROPY_FMAX)
entropy_psd = psd[:, entropy_mask]
entropy_total_power = entropy_psd.sum(axis=1, keepdims=True)
if np.any(entropy_total_power <= 0):
    raise ValueError("Cannot calculate spectral entropy from zero PSD power")

entropy_probability = entropy_psd / entropy_total_power
spectral_entropy_bits = entropy(entropy_probability, base=2, axis=1)
n_entropy_bins = entropy_probability.shape[1]
normalized_spectral_entropy = spectral_entropy_bits / np.log2(n_entropy_bins)

node_feature_matrix = db_feature_matrix.copy()
node_feature_matrix["spectral_entropy"] = normalized_spectral_entropy
node_feature_matrix.index.name = "channel"

import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.datautils.graphdataversiontwo.node_features import compute_hjorth_features

notebook_hjorth = compute_hjorth_features(mi_window)
node_feature_matrix["hjorth_mobility"] = notebook_hjorth[:, 0]
node_feature_matrix["hjorth_complexity"] = notebook_hjorth[:, 1]
assert node_feature_matrix.shape == (29, 8)
assert np.isfinite(node_feature_matrix.to_numpy()).all()
assert node_feature_matrix["spectral_entropy"].between(0.0, 1.0).all()

display(node_feature_matrix.round(4))
print(f"Node-feature matrix shape: {node_feature_matrix.shape}")
print(f"All values finite: {np.isfinite(node_feature_matrix.to_numpy()).all()}")
print(f"Entropy frequency range: {ENTROPY_FMIN:g}–{ENTROPY_FMAX:g} Hz")
print(f"Entropy frequency bins: {n_entropy_bins}")
print(
    f"Normalized entropy range: "
    f"[{normalized_spectral_entropy.min():.6f}, {normalized_spectral_entropy.max():.6f}]"
)

## Plot band-power, spectral-entropy, and Hjorth profiles in stacked panels

Eight panels retain separate scales and units for each feature.


In [ ]:
channels = node_feature_matrix.index.to_list()
channel_positions = np.arange(len(channels))

fig, axes = plt.subplots(8, 1, figsize=(14, 18), sharex=True)
for axis, band_name in zip(axes[:5], FEATURE_BANDS):
    values = node_feature_matrix[band_name].to_numpy()
    axis.plot(channel_positions, values, marker="o", linewidth=1.2, markersize=4)
    axis.fill_between(channel_positions, values, values.min(), alpha=0.15)
    axis.set_ylabel("dB")
    low_hz, high_hz = FEATURE_BANDS[band_name]
    axis.set_title(f"{band_name.title()} ({low_hz}–{high_hz} Hz)", loc="left")
    axis.grid(alpha=0.25)

entropy_values = node_feature_matrix["spectral_entropy"].to_numpy()
entropy_axis = axes[5]
entropy_axis.plot(
    channel_positions, entropy_values, marker="o", linewidth=1.2, markersize=4,
    color="tab:purple",
)
entropy_axis.fill_between(channel_positions, 0, entropy_values, alpha=0.15, color="tab:purple")
entropy_axis.set_ylabel("Normalized")
entropy_axis.set_ylim(0, 1)
entropy_axis.set_title("Spectral entropy (1–40 Hz)", loc="left")
entropy_axis.grid(alpha=0.25)

for axis, name, units in zip(axes[6:], ("hjorth_mobility", "hjorth_complexity"), ("per sample", "dimensionless")):
    axis.plot(channel_positions, node_feature_matrix[name], marker="o")
    axis.set_title(name.replace("_", " ").title(), loc="left")
    axis.set_ylabel(units)
    axis.grid(alpha=0.25)

axes[-1].set_xticks(channel_positions)
axes[-1].set_xticklabels(channels, rotation=60, ha="right")
axes[-1].set_xlabel("EEG channel")
fig.suptitle(
    f"{SUBJECT} — trial {FEATURE_TRIAL_INDEX + 1} — eight node-feature profiles"
)
fig.tight_layout()
plt.show()

# Edge Features

## Single-trial frequency-specific wPLI edge features

Node and edge features are extracted independently from the same 29-channel cleaned motor-imagery trial. Node features summarize per-channel spectral power, while edge features quantify pairwise non-zero-lag phase consistency directly from the EEG time series; wPLI is **not** calculated from the node-power or dB matrices.

For channels $i$ and $j$ in frequency band $b$, ordinary weighted Phase Lag Index is

$$
\operatorname{wPLI}_{ij}^{(b)} = \frac{\left|\mathbb{E}\left[\operatorname{Im}\left(S_{ij}^{(b)}\right)\right]\right|}{\mathbb{E}\left[\left|\operatorname{Im}\left(S_{ij}^{(b)}\right)\right|\right]},
$$

where $S_{ij}^{(b)}$ is the band-specific cross-spectrum. Thus the same trial produces node features $\mathbf{X} \in \mathbb{R}^{29\times6}$ and five symmetric wPLI adjacency matrices $\mathbf{A}^{(b)} \in [0,1]^{29\times29}$. All 406 unordered channel pairs are retained. Both directions are stored for PyTorch Geometric, giving 812 directed edges without self-loops. This is an exploratory single-trial estimate; delta-band connectivity is expected to be the least stable because a four-second window contains few low-frequency cycles.

Band averages below use explicit inclusive frequency bounds. This corrects the upper-index slicing issue in mne-connectivity 0.9.0; earlier displayed edge values may therefore differ.


In [ ]:
from mne_connectivity import spectral_connectivity_time

WPLI_BAND_NAMES = list(FEATURE_BANDS)
WPLI_FMIN = tuple(FEATURE_BANDS[name][0] for name in WPLI_BAND_NAMES)
WPLI_FMAX = tuple(FEATURE_BANDS[name][1] for name in WPLI_BAND_NAMES)

# mi_window is the same [29, samples] code-2-to-code-3 epoch used above.
trial_eeg = mi_window[np.newaxis, :, :]
wpli_connectivity = spectral_connectivity_time(
    data=trial_eeg,
    freqs=np.arange(1.0, 41.0),
    method="wpli",
    average=False,
    sfreq=eeg_raw.info["sfreq"],
    mode="multitaper",
    fmin=WPLI_FMIN,
    fmax=WPLI_FMAX,
    faverage=False,  # Average exact band bounds explicitly below.
    sm_times=0.5,
    mt_bandwidth=2.0,
    n_cycles=3.0,
    verbose=False,
)

# MNE returns [trial, source, target, band] and stores one triangle.
wpli_dense = wpli_connectivity.get_data(output="dense")[0]
# Explicit bounds avoid the mne-connectivity 0.9.0 band-averaging bug.
connectivity_freqs = np.asarray(wpli_connectivity.freqs)
wpli_dense = np.stack([
    wpli_dense[..., (connectivity_freqs >= low) & (connectivity_freqs <= high)].mean(axis=-1)
    for low, high in FEATURE_BANDS.values()
], axis=-1)
wpli_adjacency = np.empty((len(WPLI_BAND_NAMES), 29, 29), dtype=float)
for band_idx in range(len(WPLI_BAND_NAMES)):
    triangle = wpli_dense[:, :, band_idx]
    adjacency = triangle + triangle.T
    np.fill_diagonal(adjacency, 0.0)
    wpli_adjacency[band_idx] = adjacency

# Remove only negligible floating-point excursions outside the theoretical range.
tolerance = 1e-12
assert wpli_adjacency.shape == (5, 29, 29)
assert np.isfinite(wpli_adjacency).all()
assert wpli_adjacency.min() >= -tolerance
assert wpli_adjacency.max() <= 1.0 + tolerance
wpli_adjacency = np.clip(wpli_adjacency, 0.0, 1.0)
for adjacency in wpli_adjacency:
    assert np.allclose(adjacency, adjacency.T)
    assert np.allclose(np.diag(adjacency), 0.0)

print(f"Subject: {SUBJECT}")
print(f"Trial: {FEATURE_TRIAL_INDEX + 1}")
print(f"MI signal shape: {mi_window.shape}")
print(f"wPLI adjacency shape: {wpli_adjacency.shape}")
print(f"wPLI range: [{wpli_adjacency.min():.6f}, {wpli_adjacency.max():.6f}]")
print(f"All finite: {np.isfinite(wpli_adjacency).all()}")
print(f"All symmetric: {all(np.allclose(a, a.T) for a in wpli_adjacency)}")

In [ ]:
# One fixed complete graph: 406 unordered pairs, saved in both directions.
upper_source, upper_target = np.triu_indices(eeg_raw.info["nchan"], k=1)
n_undirected_edges = upper_source.size
source = np.concatenate([upper_source, upper_target])
target = np.concatenate([upper_target, upper_source])
edge_index = np.vstack([source, target]).astype(np.int64)
edge_attr = wpli_adjacency[:, source, target].T

assert n_undirected_edges == 406
assert edge_index.shape == (2, 812)
assert edge_attr.shape == (812, 5)
assert not np.any(edge_index[0] == edge_index[1])
assert np.isfinite(edge_attr).all()
assert np.allclose(edge_attr[:n_undirected_edges], edge_attr[n_undirected_edges:])

edge_table = pd.DataFrame({
    "source": [eeg_raw.ch_names[i] for i in source],
    "target": [eeg_raw.ch_names[j] for j in target],
    **{name: edge_attr[:, idx] for idx, name in enumerate(WPLI_BAND_NAMES)},
})

print(f"Unique undirected channel pairs: {n_undirected_edges}")
print(f"Directed edges stored: {edge_index.shape[1]}")
print(f"edge_index shape: {edge_index.shape}")
print(f"edge_attr shape: {edge_attr.shape}")
display(edge_table.head(10).round(4))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 11), constrained_layout=True)
axes = axes.ravel()
for band_idx, (axis, band_name) in enumerate(zip(axes, WPLI_BAND_NAMES)):
    image = axis.imshow(wpli_adjacency[band_idx], vmin=0, vmax=1, cmap="viridis")
    axis.set_title(f"{band_name.title()} {FEATURE_BANDS[band_name][0]}–{FEATURE_BANDS[band_name][1]} Hz")
    axis.set_xticks(np.arange(29))
    axis.set_yticks(np.arange(29))
    axis.set_xticklabels(eeg_raw.ch_names, rotation=90, fontsize=6)
    axis.set_yticklabels(eeg_raw.ch_names, fontsize=6)
    axis.set_xlabel("Target channel")
    axis.set_ylabel("Source channel")
axes[-1].axis("off")
colorbar = fig.colorbar(image, ax=axes.tolist(), shrink=0.8, pad=0.02)
colorbar.set_label("Ordinary wPLI")
fig.suptitle(f"{SUBJECT} — trial {FEATURE_TRIAL_INDEX + 1} — frequency-specific wPLI", fontsize=15)
plt.show()

## Separate single-trial Phase Locking Value (PLV) edge features

For comparison with wPLI, this section independently calculates frequency-specific Phase Locking Value directly from the same 29-channel motor-imagery time series. For channels $i$ and $j$ in band $b$,

$$
\operatorname{PLV}_{ij}^{(b)} = \left|\mathbb{E}\left[e^{\mathrm{i}(\phi_i^{(b)}-\phi_j^{(b)})}\right]\right| \in [0,1].
$$

PLV measures consistency of phase difference, including zero-lag coupling. Consequently, it is more sensitive than wPLI to volume conduction and common-reference effects. The PLV results are stored in separate variables and do not replace the wPLI graph.

In [ ]:
plv_connectivity = spectral_connectivity_time(
    data=trial_eeg,
    freqs=np.arange(1.0, 41.0),
    method="plv",
    average=False,
    sfreq=eeg_raw.info["sfreq"],
    mode="multitaper",
    fmin=WPLI_FMIN,
    fmax=WPLI_FMAX,
    faverage=False,  # Average exact band bounds explicitly below.
    sm_times=0.5,
    mt_bandwidth=2.0,
    n_cycles=3.0,
    verbose=False,
)

# Preserve the same band order and channel order used for wPLI.
plv_dense = plv_connectivity.get_data(output="dense")[0]
# Explicit bounds avoid the mne-connectivity 0.9.0 band-averaging bug.
connectivity_freqs = np.asarray(plv_connectivity.freqs)
plv_dense = np.stack([
    plv_dense[..., (connectivity_freqs >= low) & (connectivity_freqs <= high)].mean(axis=-1)
    for low, high in FEATURE_BANDS.values()
], axis=-1)
plv_adjacency = np.empty((len(WPLI_BAND_NAMES), 29, 29), dtype=float)
for band_idx in range(len(WPLI_BAND_NAMES)):
    triangle = plv_dense[:, :, band_idx]
    adjacency = triangle + triangle.T
    np.fill_diagonal(adjacency, 0.0)
    plv_adjacency[band_idx] = adjacency

assert plv_adjacency.shape == (5, 29, 29)
assert np.isfinite(plv_adjacency).all()
assert plv_adjacency.min() >= -tolerance
assert plv_adjacency.max() <= 1.0 + tolerance
plv_adjacency = np.clip(plv_adjacency, 0.0, 1.0)
for adjacency in plv_adjacency:
    assert np.allclose(adjacency, adjacency.T)
    assert np.allclose(np.diag(adjacency), 0.0)

print(f"Subject: {SUBJECT}")
print(f"Trial: {FEATURE_TRIAL_INDEX + 1}")
print(f"PLV adjacency shape: {plv_adjacency.shape}")
print(f"PLV range: [{plv_adjacency.min():.6f}, {plv_adjacency.max():.6f}]")
print(f"All finite: {np.isfinite(plv_adjacency).all()}")
print(f"All symmetric: {all(np.allclose(a, a.T) for a in plv_adjacency)}")

In [ ]:
# Reuse the exact same complete directed edge list as the wPLI graph.
plv_edge_index = edge_index.copy()
plv_edge_attr = plv_adjacency[:, source, target].T

assert np.array_equal(plv_edge_index, edge_index)
assert plv_edge_index.shape == (2, 812)
assert plv_edge_attr.shape == (812, 5)
assert np.isfinite(plv_edge_attr).all()
assert np.allclose(
    plv_edge_attr[:n_undirected_edges],
    plv_edge_attr[n_undirected_edges:],
)

plv_edge_table = pd.DataFrame({
    "source": [eeg_raw.ch_names[i] for i in source],
    "target": [eeg_raw.ch_names[j] for j in target],
    **{name: plv_edge_attr[:, idx] for idx, name in enumerate(WPLI_BAND_NAMES)},
})

print(f"PLV edge_index shape: {plv_edge_index.shape}")
print(f"PLV edge_attr shape: {plv_edge_attr.shape}")
display(plv_edge_table.head(10).round(4))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 11), constrained_layout=True)
axes = axes.ravel()
for band_idx, (axis, band_name) in enumerate(zip(axes, WPLI_BAND_NAMES)):
    image = axis.imshow(plv_adjacency[band_idx], vmin=0, vmax=1, cmap="magma")
    axis.set_title(f"{band_name.title()} {FEATURE_BANDS[band_name][0]}–{FEATURE_BANDS[band_name][1]} Hz")
    axis.set_xticks(np.arange(29))
    axis.set_yticks(np.arange(29))
    axis.set_xticklabels(eeg_raw.ch_names, rotation=90, fontsize=6)
    axis.set_yticklabels(eeg_raw.ch_names, fontsize=6)
    axis.set_xlabel("Target channel")
    axis.set_ylabel("Source channel")
axes[-1].axis("off")
colorbar = fig.colorbar(image, ax=axes.tolist(), shrink=0.8, pad=0.02)
colorbar.set_label("Phase Locking Value")
fig.suptitle(f"{SUBJECT} — trial {FEATURE_TRIAL_INDEX + 1} — frequency-specific PLV", fontsize=15)
plt.show()

## Separate single-trial imaginary coherence (iCoh) edge features

This section calculates frequency-specific imaginary coherence directly from the same 29-channel motor-imagery time series. For channels $i$ and $j$,

$$
\operatorname{iCoh}_{ij}(f) = \operatorname{Im}\left(\frac{S_{ij}(f)}{\sqrt{S_{ii}(f)S_{jj}(f)}}\right).
$$

Signed iCoh is antisymmetric: $\operatorname{iCoh}_{ji}=-\operatorname{iCoh}_{ij}$. The signed triangular MNE result is retained as `icoh_signed_dense`, while the undirected graph uses $|\operatorname{iCoh}|$, which is symmetric and lies in $[0,1]$. Like wPLI, iCoh suppresses zero-lag coupling, but unlike wPLI it retains the magnitude of normalized imaginary coherency.

In [ ]:
icoh_connectivity = spectral_connectivity_time(
    data=trial_eeg,
    freqs=np.arange(1.0, 41.0),
    method="imcoh",
    average=False,
    sfreq=eeg_raw.info["sfreq"],
    mode="multitaper",
    fmin=WPLI_FMIN,
    fmax=WPLI_FMAX,
    faverage=False,  # Average exact band bounds explicitly below.
    sm_times=0.5,
    mt_bandwidth=2.0,
    n_cycles=3.0,
    verbose=False,
)

# MNE stores one signed triangle; use its magnitude for an undirected graph.
icoh_signed_dense = icoh_connectivity.get_data(output="dense")[0]
# Explicit bounds avoid the mne-connectivity 0.9.0 band-averaging bug.
connectivity_freqs = np.asarray(icoh_connectivity.freqs)
icoh_signed_dense = np.stack([
    icoh_signed_dense[..., (connectivity_freqs >= low) & (connectivity_freqs <= high)].mean(axis=-1)
    for low, high in FEATURE_BANDS.values()
], axis=-1)
icoh_adjacency = np.empty((len(WPLI_BAND_NAMES), 29, 29), dtype=float)
for band_idx in range(len(WPLI_BAND_NAMES)):
    magnitude_triangle = np.abs(icoh_signed_dense[:, :, band_idx])
    adjacency = magnitude_triangle + magnitude_triangle.T
    np.fill_diagonal(adjacency, 0.0)
    icoh_adjacency[band_idx] = adjacency

assert icoh_signed_dense.shape == (29, 29, 5)
assert np.isfinite(icoh_signed_dense).all()
assert icoh_adjacency.shape == (5, 29, 29)
assert np.isfinite(icoh_adjacency).all()
assert icoh_adjacency.min() >= -tolerance
assert icoh_adjacency.max() <= 1.0 + tolerance
icoh_adjacency = np.clip(icoh_adjacency, 0.0, 1.0)
for adjacency in icoh_adjacency:
    assert np.allclose(adjacency, adjacency.T)
    assert np.allclose(np.diag(adjacency), 0.0)

print(f"Subject: {SUBJECT}")
print(f"Trial: {FEATURE_TRIAL_INDEX + 1}")
print(f"Signed iCoh triangle range: [{icoh_signed_dense.min():.6f}, {icoh_signed_dense.max():.6f}]")
print(f"Absolute iCoh adjacency shape: {icoh_adjacency.shape}")
print(f"Absolute iCoh range: [{icoh_adjacency.min():.6f}, {icoh_adjacency.max():.6f}]")
print(f"All finite: {np.isfinite(icoh_adjacency).all()}")
print(f"All symmetric: {all(np.allclose(a, a.T) for a in icoh_adjacency)}")

In [ ]:
# Reuse the same complete directed edge list as wPLI and PLV.
icoh_edge_index = edge_index.copy()
icoh_edge_attr = icoh_adjacency[:, source, target].T

assert np.array_equal(icoh_edge_index, edge_index)
assert icoh_edge_index.shape == (2, 812)
assert icoh_edge_attr.shape == (812, 5)
assert np.isfinite(icoh_edge_attr).all()
assert np.allclose(
    icoh_edge_attr[:n_undirected_edges],
    icoh_edge_attr[n_undirected_edges:],
)

icoh_edge_table = pd.DataFrame({
    "source": [eeg_raw.ch_names[i] for i in source],
    "target": [eeg_raw.ch_names[j] for j in target],
    **{name: icoh_edge_attr[:, idx] for idx, name in enumerate(WPLI_BAND_NAMES)},
})

print(f"iCoh edge_index shape: {icoh_edge_index.shape}")
print(f"iCoh edge_attr shape: {icoh_edge_attr.shape}")
display(icoh_edge_table.head(10).round(4))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 11), constrained_layout=True)
axes = axes.ravel()
for band_idx, (axis, band_name) in enumerate(zip(axes, WPLI_BAND_NAMES)):
    image = axis.imshow(icoh_adjacency[band_idx], vmin=0, vmax=1, cmap="cividis")
    axis.set_title(f"{band_name.title()} {FEATURE_BANDS[band_name][0]}–{FEATURE_BANDS[band_name][1]} Hz")
    axis.set_xticks(np.arange(29))
    axis.set_yticks(np.arange(29))
    axis.set_xticklabels(eeg_raw.ch_names, rotation=90, fontsize=6)
    axis.set_yticklabels(eeg_raw.ch_names, fontsize=6)
    axis.set_xlabel("Target channel")
    axis.set_ylabel("Source channel")
axes[-1].axis("off")
colorbar = fig.colorbar(image, ax=axes.tolist(), shrink=0.8, pad=0.02)
colorbar.set_label("Absolute imaginary coherence")
fig.suptitle(f"{SUBJECT} — trial {FEATURE_TRIAL_INDEX + 1} — frequency-specific |iCoh|", fontsize=15)
plt.show()

## Separate four-feature node representation

Select one band with `SELECTED_NODE_BAND` (alpha by default; choose delta, theta, alpha, beta, or gamma). Each electrode has **one band power in dB, normalized spectral entropy, Hjorth mobility, and Hjorth complexity**, giving a `(29, 4)` matrix.

This section uses the same EEG trial (`mi_window`) as the earlier node features. Only the power column changes with the selected band: entropy remains over 1–40 Hz, and Hjorth features use the original trial signal without additional band filtering. This is an alternative node representation for inspection.

With first and second sample differences, mobility is `sqrt(var(diff(x)) / var(x))`; complexity is `sqrt(var(diff(x, 2)) / var(diff(x))) / mobility`. Differences are not multiplied by sampling frequency, so mobility is expressed per sample and complexity is dimensionless. Constant signals or zero first-difference variance make these ratios undefined and are reported explicitly.


In [ ]:
SELECTED_NODE_BAND = "alpha"
if SELECTED_NODE_BAND not in FEATURE_BANDS:
    raise ValueError(f"Select a band from {tuple(FEATURE_BANDS)}")

selected_hjorth = compute_hjorth_features(mi_window)
hjorth_mobility = selected_hjorth[:, 0]
hjorth_complexity = selected_hjorth[:, 1]

four_feature_matrix = pd.DataFrame({
    f"{SELECTED_NODE_BAND}_power_db": db_feature_matrix[SELECTED_NODE_BAND],
    "spectral_entropy": normalized_spectral_entropy,
    "hjorth_mobility": hjorth_mobility,
    "hjorth_complexity": hjorth_complexity,
}, index=db_feature_matrix.index)
four_feature_matrix.index.name = "channel"

assert four_feature_matrix.index.to_list() == eeg_raw.ch_names
assert four_feature_matrix.shape == (29, 4)
assert np.isfinite(four_feature_matrix.to_numpy()).all()
assert four_feature_matrix["spectral_entropy"].between(0, 1).all()
display(four_feature_matrix.round(4))
print(f"Selected band: {SELECTED_NODE_BAND}; node-feature matrix: {four_feature_matrix.shape}")


## Plot band-power and spectral-entropy profiles in stacked panels

Four stacked panels show the selected band power, spectral entropy, Hjorth mobility, and Hjorth complexity across the same 29 electrodes. Each panel keeps its own units and vertical scale; the horizontal axis is electrode order, not time.


In [ ]:
import matplotlib.pyplot as plt

four_feature_positions = np.arange(len(four_feature_matrix))
four_feature_labels = [
    (f"{SELECTED_NODE_BAND.title()} band power", "dB re 1 µV²"),
    ("Spectral entropy (1–40 Hz)", "normalized entropy"),
    ("Hjorth mobility", "per sample"),
    ("Hjorth complexity", "dimensionless"),
]
fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True, constrained_layout=True)
for axis, column, (title, units) in zip(axes, four_feature_matrix.columns, four_feature_labels):
    axis.plot(four_feature_positions, four_feature_matrix[column], marker="o", linewidth=1.2)
    axis.set_title(title, loc="left")
    axis.set_ylabel(units)
    axis.grid(alpha=0.3)
axes[1].set_ylim(0, 1)
axes[-1].set_xticks(four_feature_positions)
axes[-1].set_xticklabels(four_feature_matrix.index, rotation=90)
axes[-1].set_xlabel("EEG electrode")
fig.suptitle(f"{SUBJECT} — trial {FEATURE_TRIAL_INDEX + 1} — four node features")
plt.show()


## Verify the existing edge features for the selected trial

This section checks the earlier **EEG** wPLI, PLV, and absolute iCoh calculations across all five bands. Each method stores `(812, 5)` attributes: 406 unique electrode pairs and their reverse entries, with one value per band. These mirrored entries do not imply directed physiological connectivity.

Checks cover finite values in [0, 1], no self-loops, complete pair coverage, channel/band order, adjacency symmetry, reverse-edge equality, and reconstruction of adjacency from edge attributes. A fresh call to version two's shared implementation checks agreement with the notebook's separate method calls on exactly the same trial. Both paths use MNE, so this is an implementation-consistency check, not an independent proof of the connectivity estimators.

This verifies one selected EEG trial, not all subjects, saved datasets, or CSD edges. No feature definitions or dataset plan are changed. Run the earlier node and edge calculation cells first.


In [ ]:
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.datautils.graphdataversiontwo import config as edge_check_config
from src.datautils.graphdataversiontwo.edge_features import (
    build_edge_index as checked_build_edge_index,
    build_edge_features as checked_build_edge_features,
    compute_connectivity as checked_compute_connectivity,
)

assert tuple(eeg_raw.ch_names) == edge_check_config.CHANNEL_NAMES
assert tuple(FEATURE_BANDS) == edge_check_config.BAND_NAMES
assert tuple(FEATURE_BANDS.values()) == edge_check_config.BANDS
np.testing.assert_array_equal(trial_eeg[0], mi_window)
np.testing.assert_array_equal(
    mi_window, eeg_raw.get_data(start=feature_start, stop=feature_stop)
)

checked_index = checked_build_edge_index(len(eeg_raw.ch_names))
assert checked_index.shape == (2, 812)
assert np.unique(checked_index, axis=1).shape[1] == 812
assert not np.any(checked_index[0] == checked_index[1])
np.testing.assert_array_equal(checked_index[:, 406:], checked_index[::-1, :406])
for notebook_index in (edge_index, plv_edge_index, icoh_edge_index):
    np.testing.assert_array_equal(notebook_index, checked_index)

edge_check_input_copy = mi_window.copy()
checked_connectivity = checked_compute_connectivity(mi_window, eeg_raw.info["sfreq"])
checked_attributes = checked_build_edge_features(checked_connectivity, checked_index)
np.testing.assert_array_equal(mi_window, edge_check_input_copy)

notebook_edge_variants = {
    "wpli": (wpli_adjacency, edge_attr),
    "plv": (plv_adjacency, plv_edge_attr),
    "icoh_abs": (icoh_adjacency, icoh_edge_attr),
}
edge_verification_rows = []
for method, (adjacency, attributes) in notebook_edge_variants.items():
    assert adjacency.shape == (5, 29, 29)
    assert attributes.shape == (812, 5)
    assert np.isfinite(adjacency).all() and np.isfinite(attributes).all()
    assert np.all((attributes >= 0) & (attributes <= 1))
    np.testing.assert_allclose(adjacency, adjacency.transpose(0, 2, 1), rtol=0, atol=1e-12)
    np.testing.assert_array_equal(np.diagonal(adjacency, axis1=1, axis2=2), 0)
    np.testing.assert_allclose(attributes[:406], attributes[406:], rtol=0, atol=1e-12)
    rebuilt = np.zeros_like(adjacency)
    rebuilt[:, checked_index[0], checked_index[1]] = attributes.T
    np.testing.assert_allclose(rebuilt, adjacency, rtol=0, atol=1e-12)
    np.testing.assert_allclose(adjacency, checked_connectivity[method], rtol=1e-7, atol=1e-10)
    np.testing.assert_allclose(attributes, checked_attributes[method], rtol=1e-7, atol=1e-10)
    for band_index, band_name in enumerate(FEATURE_BANDS):
        unique_values = attributes[:406, band_index]
        edge_verification_rows.append({
            "method": method, "band": band_name, "unique_pairs": len(unique_values),
            "min": unique_values.min(), "max": unique_values.max(),
            "mean": unique_values.mean(), "std": unique_values.std(),
            "max_abs_package_difference": np.max(np.abs(
                attributes[:, band_index] - checked_attributes[method][:, band_index]
            )),
            "status": "PASS",
        })

edge_verification_report = pd.DataFrame(edge_verification_rows)
print(edge_verification_report.to_string(index=False))
print(f"Verified {SUBJECT}, trial {FEATURE_TRIAL_INDEX + 1}, "
      f"samples [{feature_start}, {feature_stop}), EEG only.")
print("PASS: all three methods × five bands; 406 unique pairs / 812 stored edges per method.")
print("Scope: structural checks and notebook/package agreement; no CSD or dataset-wide validation.")
